# Shreve Week 02 — Random Walk & Brownian Motion

**Week 2** — Random Walk & Brownian Motion

This notebook teaches **random walk & brownian motion** in the style of our graduate probability notebook: precise definitions from Shreve, then **verified with Python**.

## What you will learn

| Part | Topic | Shreve |
|------|-------|--------|
| 1 | **Symmetric random walk** | Ch. 3.2 |
| 2 | **Quadratic variation of \(W\)** | Ch. 3.2 |
| 3 | **Donsker's theorem / construction** | Ch. 3.3 |
| 4 | **Covariance & Gaussian paths** | Ch. 3.2 |
| — | **Problem set** | Ch. 3 exercises |

## How to use this notebook

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — stochastic calculus is about *relationships*, not memorized formulas.
3. Sections end with **"Your turn"** exercises. The **problem set** at the end has **click-to-reveal solutions**.
4. Primary reference: **Shreve**, *Stochastic Calculus for Finance II* — see chapter pointers in each section.

**Shreve reference:** Ch. 3 — [Vol II PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Steve_ShreveStochastic_Calculus_for_Finance_II.pdf)

Let's begin.


---
## Setup — run this first

We use NumPy for simulation, SciPy for exact distributions, and Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import erf

np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", stats.__version__)


---
# Part 1 — Symmetric Random Walk

\(S_0 = 0\), \(S_n = \sum_{i=1}^n X_i\) with \(P(X_i = \pm 1) = 1/2\).

Properties: \(E[S_n] = 0\), \(\text{Var}(S_n) = n\), increments independent.

**Shreve Ch. 3.2:** random walk as discrete Brownian motion.


In [ ]:
rng = np.random.default_rng(10)
n_paths, n = 100, 500
walks = rng.choice([-1, 1], size=(n_paths, n))
paths = np.cumsum(walks, axis=1)

fig, ax = plt.subplots()
for p in paths[:30]:
    ax.plot(p, alpha=0.5)
ax.set_title("Symmetric random walk paths")
plt.show()

# Distribution of S_n
n = 100
S_n = rng.choice([-1, 1], size=(200_000, n)).sum(axis=1)
print(f"S_{n}: mean={S_n.mean():.3f}, var={S_n.var():.1f} (theory 0, {n})")


---
# Part 2 — Quadratic Variation

For Brownian motion \(W_t\), **quadratic variation** on \([0,T]\):
$$[W, W]_T = \lim_{n\to\infty} \sum_{i=1}^n (W_{t_i} - W_{t_{i-1}})^2 = T$$

For smooth functions, quadratic variation is 0; for \(W\), it is \(T\). This is why \((dW)^2 = dt\) in Itô calculus.

**Shreve Ch. 3.2:** \(W\) has non-zero quadratic variation.


In [ ]:
def quadratic_variation_demo(T=1.0, n_list=(10, 100, 1000, 10000), seed=11):
    rng = np.random.default_rng(seed)
    print("Quadratic variation of Brownian motion on [0,T]")
    for n in n_list:
        dt = T / n
        dW = rng.normal(0, np.sqrt(dt), size=n)
        W = np.cumsum(dW)
        qv = np.sum(dW**2)
        print(f"  n={n:5d}: QV = {qv:.4f} (theory T={T})")

quadratic_variation_demo()


---
# Part 3 — Construction via Scaled Random Walk

**Donsker's theorem:** \(W_t^{(n)} = \frac{1}{\sqrt{n}} S_{\lfloor nt \rfloor}\) converges in distribution to \(W_t\) as \(n \to \infty\).

**Shreve Ch. 3.3:** rigorous construction of Brownian motion.


In [ ]:
def donsker_path(T=1.0, n=5000, seed=12):
    rng = np.random.default_rng(seed)
    dt = T / n
    dW = rng.choice([-1, 1], size=n) / np.sqrt(n)
    W_rw = np.concatenate([[0], np.cumsum(dW)])
    dW_bm = rng.normal(0, np.sqrt(dt), size=n)
    W_bm = np.concatenate([[0], np.cumsum(dW_bm)])
    t = np.linspace(0, T, n + 1)
    fig, ax = plt.subplots()
    ax.plot(t, W_rw, label="scaled RW", alpha=0.8)
    ax.plot(t, W_bm, label="Brownian", alpha=0.8)
    ax.legend()
    ax.set_title("Scaled random walk vs simulated Brownian motion")
    plt.show()

donsker_path()


---
# Part 4 — Covariance Structure

Brownian motion satisfies:
- \(W_0 = 0\)
- \(W_t - W_s \sim N(0, t-s)\) for \(t > s\)
- \(E[W_t W_s] = \min(t, s)\)

**Shreve Ch. 3.2:** characterizing \(W\) by independent Gaussian increments.


In [ ]:
rng = np.random.default_rng(13)
T, n = 1.0, 1000
dt = T / n
dW = rng.normal(0, np.sqrt(dt), size=n)
W = np.concatenate([[0], np.cumsum(dW)])
t = np.linspace(0, T, n + 1)

# Check E[W_t W_s] = min(t,s) at sample times
idx = [0, n//4, n//2, 3*n//4, n]
times = t[idx]
Ws = W[idx]
print("Covariance matrix (simulated vs theory min(t,s)):")
cov_sim = np.cov(Ws)
cov_th = np.minimum.outer(times, times)
print("Simulated:", np.round(cov_sim, 4))
print("Theory:   ", np.round(cov_th, 4))


**Your turn:** Simulate \(W_t\) at \(t=1\) many times. Plot the histogram and overlay \(N(0,1)\).


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. For symmetric RW \(S_n\), find \(E[S_n^4]\) (hint: use \(S_{n+1} = S_n + X_{n+1}\)).

2. Show \(E[W_t W_s] = \min(t,s)\) for Brownian motion using independent increments.

3. Explain why \(\sum (\Delta W_i)^2 \to T\) but \(\sum |\Delta W_i| \to \infty\) as mesh goes to 0.

4. If \(W_t^{(n)}\) is scaled RW, what is \(\text{Var}(W_t^{(n)})\)?


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** \(E[S_{n+1}^4] = E[(S_n+X)^4]\); cross terms vanish; yields \(E[S_n^4] = 3n^2 + n\).

**2.** Write \(W_t = W_s + (W_t - W_s)\); cross term \(E[W_s(W_t-W_s)] = W_s E[W_t-W_s] = 0\); so \(E[W_t W_s] = E[W_s^2] = s\) for \(t>s\).

**3.** QV: each squared increment has mean \(dt\), sum → \(T\). Total variation: \(E[|\Delta W|] \sim \sqrt{dt}\), sum \(\sim n\sqrt{dt} = T/\sqrt{dt} \to \infty\).

**4.** \(\text{Var}(W_t^{(n)}) = t\) (each step variance \(1/n\), \(nt\) steps).

</details>


---
## Further reading

- **Shreve**, *Stochastic Calculus for Finance II*, Ch. 3 — primary text for this week.
- **Shreve**, *Stochastic Calculus for Finance I* — discrete-time foundations (Ch. 1–5).
- **Karatzas & Shreve**, *Brownian Motion and Stochastic Calculus* — rigorous continuous-time theory.

Whenever a theorem says a process "converges" or a formula "holds in expectation," you can **simulate it** here and see the numbers match.
